# 11 · What-if rescheduling (Step 8)

At 20:00 on D−1 the desk sees the day-D risk of every flight. For the **daily top 10%** (the primary policy) it tries moving the scheduled departure by whole hours on the same day and keeps a move only if the model's **predicted** number of delayed flights at that origin on D falls by at least δ, counting the effect on every other flight at that origin (congestion spill-over).

**Explicit assumptions.**

* **A1, predictive not causal.** The model's association between hour / congestion and delay is read as the effect of a move. Weather and ATC programmes are unknown at 20:00 D−1.
* **A2, scope of a move.** Same day, whole hours, block time unchanged; carrier, aircraft, route and flight number unchanged; rotations, crews and connections not modelled.
* **A3, capacity.** No origin-hour busier than that origin's busiest training hour; new hours within its observed departure hours.
* **A4, cost.** Each move costs 0.02 expected delayed flights; the window (1–2 h) and δ are chosen on **Sep–Oct** by net predicted benefit and frozen for Nov–Dec.

**Leakage.** A move edits only schedule-derived inputs and the moved flight's origin-hour history, read from a table built with the Phase 1 decision clock; tests show that scrambling every outcome from day D on leaves day D's plan unchanged (`tests/test_reschedule.py`).

In [ ]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().resolve().parent
for p in (ROOT / 'src', ROOT.parent / 'phase1' / 'src'):
    sys.path.insert(0, str(p))

import aerocast_dl  # torch before XGBoost (OpenMP)
import numpy as np, pandas as pd
from IPython.display import Image, display
from aerocast_dl import config, pipeline
R, F = config.RESULTS_DIR, config.FIGURES_DIR
pd.set_option('display.precision', 4)


## 1. Rule chosen on Sep–Oct

In [ ]:
sel = json.loads((R / 'selection.json').read_text())
print('model:', sel['reschedule']['model'], '| frozen rule:', sel['reschedule']['params'], '| move cost:', sel['reschedule']['move_cost'])
pd.read_csv(R / 'reschedule_tuning.csv')

## 2. Applied to Nov–Dec, with cost sensitivity and a naive comparison

The naive rule moves every candidate one hour earlier whenever capacity allows. Comparing both shows whether the model's per-flight assessment adds anything beyond 'earlier is better'.

In [ ]:
pd.read_csv(R / 'reschedule_test.csv')

In [ ]:
mv = pd.read_csv(R / 'reschedule_test_moves.csv')
print(f"{len(mv)} moves; mean risk of moved flights {mv['risk_before'].mean():.3f} -> {mv['risk_after'].mean():.3f}; spill-over share {mv['spillover'].sum() / mv['net_gain'].sum():.2f}")
mv.groupby(['origin', 'shift']).size().unstack(fill_value=0)

In [ ]:
display(Image(F / 'reschedule_shifts.png', width=420))

## 3. How to read this

The numbers are **predicted** delays avoided under A1–A4, not realised savings: the data hold no counterfactual outcomes, no costs and no rotations. The analysis answers a narrower question: *given this model, which re-timings would a capacity-limited desk consider, and how much predicted risk would they remove?*